# Point and patch preconditioners for Poisson's equation

How much do we gain by correcting several neighboring coefficients together? How much do we gain by updating the residual between local corrections? We change each choice separately and compare iteration counts for preconditioned conjugate gradients (PCG) on the meshes of Chapter 7.

On the unit square $\Omega$, let $V_h\subset H^1_0(\Omega)$ consist of continuous, piecewise affine functions. We seek $u_h\in V_h$ satisfying

$$
\begin{align*}
\int_\Omega \nabla u_h\cdot\nabla v_h\,dx
&=\int_\Omega f v_h\,dx \qquad\text{for all }v_h\in V_h,\\
f(x,y)&=2\bigl(x(1-x)+y(1-y)\bigr).
\end{align*}
$$

The coarsest mesh splits $3\times3$ squares along alternating diagonals. Each uniform refinement halves the mesh size. The first two refinements bring us to 121 free unknowns.

In [ ]:
import matplotlib.pyplot as plt
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element1D, Element2D
from ngsolve import *
from ngsolve.webgui import Draw

levels_to_compare = [2, 3, 4]
tolerance = 1e-8

In [ ]:
def chapter_mesh():
    n = 3
    mesh = NetgenMesh(dim=2)
    domain = mesh.AddRegion("domain", dim=2)
    boundary = {side: mesh.AddRegion(side, dim=1)
                for side in ["left", "right", "bottom", "top"]}
    points = {(i, j): mesh.Add(MeshPoint(Pnt(i / n, j / n, 0)))
              for j in range(n + 1) for i in range(n + 1)}

    for j in range(n):
        for i in range(n):
            bl, br = points[i, j], points[i + 1, j]
            tl, tr = points[i, j + 1], points[i + 1, j + 1]
            if (i + j) % 2 == 0:
                triangles = [[bl, br, tr], [bl, tr, tl]]
            else:
                triangles = [[bl, br, tl], [br, tr, tl]]
            for triangle in triangles:
                mesh.Add(Element2D(domain, triangle))
    for i in range(n):
        mesh.Add(Element1D([points[i, 0], points[i + 1, 0]], index=boundary["bottom"]))
        mesh.Add(Element1D([points[i + 1, n], points[i, n]], index=boundary["top"]))
    for j in range(n):
        mesh.Add(Element1D([points[0, j + 1], points[0, j]], index=boundary["left"]))
        mesh.Add(Element1D([points[n, j], points[n, j + 1]], index=boundary["right"]))
    return Mesh(mesh)

mesh = chapter_mesh()
Draw(mesh);

## What belongs to a vertex patch?

Pick one vertex and mark every triangle touching it. Our block contains the free coefficients at all vertices of those triangles. Two neighboring patches therefore share unknowns. The view below marks one patch on the first refined mesh.

In [ ]:
view_mesh = chapter_mesh()
view_mesh.Refine()
center = min(view_mesh.vertices,
             key=lambda v: (v.point[0] - 0.5)**2 + (v.point[1] - 0.5)**2)
cell_space = L2(view_mesh, order=0)
patch_indicator = GridFunction(cell_space)
for element in center.elements:
    cell_dof, = cell_space.GetDofNrs(element)
    patch_indicator.vec[cell_dof] = 1

Draw(patch_indicator, view_mesh, name="Triangles in one vertex patch", order=1);

## Richardson with point Jacobi and Gauss–Seidel

Work on the free degrees of freedom. Write $A=D+L+L^\top$, where $D$ is the diagonal and $L$ the strictly lower triangular part of $A$. For a preconditioner $C$, Richardson uses

$$
\begin{align*}
r^{(k)} &= b-AU^{(k)}, &
Cz^{(k)} &= r^{(k)}, &
U^{(k+1)} &= U^{(k)}+\omega z^{(k)}.
\end{align*}
$$

With $C=I$ this is ordinary Richardson. Point Jacobi and forward Gauss–Seidel give

$$
\begin{align*}
C_{\rm J} &= D, &
U^{(k+1)} &= U^{(k)}+\omega D^{-1}r^{(k)},\\
C_{\rm F} &= D+L, &
U^{(k+1)} &= U^{(k)}+(D+L)^{-1}r^{(k)}.
\end{align*}
$$

Jacobi computes all corrections from the same residual. In the forward triangular solve, each row uses the corrections already found in preceding rows. The usual undamped sweep has $\omega=1$.

For **symmetric Gauss–Seidel (SGS)**, follow the forward sweep by a backward sweep using the remaining residual. For one incoming residual $r$ the two solves are

$$
\begin{align*}
(D+L)z_f &= r, &
(D+L^\top)z_b &= r-Az_f.
\end{align*}
$$

Since $r-Az_f=-L^\top z_f$, we have $(D+L^\top)(z_f+z_b)=Dz_f$. Thus the combined Richardson step and its symmetric positive definite preconditioner are

$$
\begin{align*}
U^{(k+1)} &= U^{(k)}+z_f+z_b, &
C_{\rm SGS} &= (D+L)D^{-1}(D+L^\top).
\end{align*}
$$

## Richardson with overlapping blocks

Let $R_i$ extract the free coefficients of patch $i$. Then $A_i=R_i A R_i^\top$ is its local stiffness matrix and $B_i=R_i^\top A_i^{-1}R_i$ maps a residual to a patch correction on the full free space.

**Additive patch Jacobi** solves every patch problem with the same residual and adds the corrections, including all contributions to shared coefficients,

$$
\begin{align*}
C_{\rm PJ}^{-1} &= \sum_{i=1}^m B_i, &
U^{(k+1)} &= U^{(k)}+\omega\sum_{i=1}^m B_i r^{(k)}.
\end{align*}
$$

Overlap matters for damping. Standalone Richardson with this SPD inverse action converges for $0<\omega<2/\lambda_{\max}(C_{\rm PJ}^{-1}A)$. In the PCG experiment below, PCG chooses its own step lengths.

**Symmetric patch Gauss–Seidel** uses the same blocks. Number them in sweep order. Start at $z=0$, take $r=r^{(k)}$, and update $z$ after every patch according to

$$
\begin{align*}
z &\leftarrow z+B_i(r-Az)
\qquad\text{for }i=1,\ldots,m,\;m,\ldots,1.
\end{align*}
$$

The forward pass alone gives a block Gauss–Seidel sweep. After both passes, $z=C_{\rm PSGS}^{-1}r$, and the undamped Richardson step is $U^{(k+1)}=U^{(k)}+z$. Singleton patches recover the corresponding point methods.

In [ ]:
def vertex_patches(mesh, V):
    free = V.FreeDofs()
    blocks = []
    for vertex in mesh.vertices:
        patch = set()
        for element in vertex.elements:
            for dof in V.GetDofNrs(element):
                if dof >= 0 and free[dof]:
                    patch.add(dof)
        if patch:
            blocks.append(sorted(patch))
    return blocks

## How many iterations does each choice require?

Each method starts with $U^{(0)}=0$. The following PCG loop updates its search direction from the recursive residual. For the stopping test we explicitly recompute the relative Euclidean residual of the free system,

$$
\begin{align*}
\frac{\|b-AU^{(k)}\|_2}{\|b\|_2} \le 10^{-8}.
\end{align*}
$$

This extra matrix application gives the same observable for all preconditioners.

In [ ]:
def pcg(A, b, C_inverse, tol, max_steps=200):
    U = b.CreateVector()
    U[:] = 0
    r = b.Copy()
    initial_norm = Norm(r)
    if initial_norm == 0:
        return U, [0.0]

    z = b.CreateVector()
    z.data = C_inverse * r
    d = z.Copy()
    rho = InnerProduct(r, z)
    residuals = [1.0]
    q = b.CreateVector()

    for k in range(max_steps):
        q.data = A * d
        tau = rho / InnerProduct(d, q)
        U.data += tau * d
        r.data -= tau * q

        residuals.append(Norm(b - A * U) / initial_norm)
        if residuals[-1] <= tol:
            return U, residuals

        z.data = C_inverse * r
        rho_new = InnerProduct(r, z)
        d = (z + (rho_new / rho) * d).Evaluate()
        rho = rho_new

    raise RuntimeError("PCG reached its iteration limit")

## Richardson with one multigrid V-cycle

On level $\ell$, write $A_\ell=D_\ell+L_\ell+L_\ell^\top$ and let $P_\ell$ prolongate from level $\ell-1$ to level $\ell$. Define $V_\ell(r)$ as the correction returned by one cycle starting from zero. On the coarsest level, $V_0(r)=A_0^{-1}r$ is a direct solve. On each finer level, the three corrections use the latest residual,

$$
\begin{align*}
z_1 &= (D_\ell+L_\ell)^{-1}r
&&\text{(forward point GS)},\\
r_c &= P_\ell^\top(r-A_\ell z_1)
&&\text{(restrict the remaining residual)},\\
z_2 &= z_1+P_\ell V_{\ell-1}(r_c)
&&\text{(one coarse cycle, then prolongate)},\\
V_\ell(r) &= z_2+(D_\ell+L_\ell^\top)^{-1}(r-A_\ell z_2)
&&\text{(backward point GS)}.
\end{align*}
$$

On the finest level $J$, the inverse action is $C_{\rm MG}^{-1}r=V_J(r)$. One standalone V-cycle therefore gives the Richardson update

$$
\begin{align*}
U^{(k+1)} &= U^{(k)}+V_J\bigl(b-AU^{(k)}\bigr).
\end{align*}
$$

**How to read the code below.** Every measured run uses PCG as the outer solver. It applies the selected inverse action to form $z^{(k)}=C^{-1}r^{(k)}$, then builds a conjugate direction and chooses the step length. The table therefore counts PCG iterations. The code variable `P` projects onto free coefficients and serves as the identity there. On these meshes $D=4I$ on the free space, so CG and point-Jacobi PCG have identical iterates in exact arithmetic.

For both point and patch smoothers, `GS=True` selects symmetric Gauss–Seidel. Registering the [multigrid V-cycle](00_two_grid_and_multigrid.ipynb) before the first assembly makes the whole hierarchy available.

In [ ]:
mesh = chapter_mesh()
V = H1(mesh, order=1, dirichlet=".*", autoupdate=True)
u, v = V.TnT()
a = BilinearForm(grad(u) * grad(v) * dx, symmetric=True)
f = 2 * (x * (1 - x) + y * (1 - y))
load = LinearForm(f * v * dx(bonus_intorder=4))
mg = Preconditioner(a, "multigrid", smoother="point", smoothingsteps=1,
                    cycle=1, inverse="sparsecholesky")

histories = {}
for level in range(max(levels_to_compare) + 1):
    if level > 0:
        mesh.Refine()
    a.Assemble()
    load.Assemble()
    if level in levels_to_compare:
        free = V.FreeDofs()
        P = Projector(free, True)
        A = P @ a.mat @ P
        b = (P * load.vec).Evaluate()
        blocks = vertex_patches(mesh, V)
        preconditioners = {
            "CG": P,
            "Point Jacobi": a.mat.CreateSmoother(free),
            "Point SGS": a.mat.CreateSmoother(free, GS=True),
            "Patch Jacobi": a.mat.CreateBlockSmoother(blocks),
            "Patch SGS": a.mat.CreateBlockSmoother(blocks, GS=True),
            "V-cycle": mg.mat,
        }
        row = {}
        for name, pre in preconditioners.items():
            _, residuals = pcg(A, b, pre, tolerance, max_steps=1000)
            row[name] = residuals
        histories[free.NumSet()] = row

In [ ]:
print(f"{'Free dofs':>10}" + "".join(f"{name:>14}" for name in preconditioners))
for ndof, row in histories.items():
    print(f"{ndof:10d}" + "".join(f"{len(history) - 1:14d}" for history in row.values()))

The left panel compares convergence on the finest mesh. The right panel follows each method under refinement. CG shares its plotted line with point Jacobi. **SGS** denotes one symmetric Gauss–Seidel application.

In [ ]:
fig, (ax_res, ax_it) = plt.subplots(1, 2, figsize=(10, 3.8), layout="constrained")
sizes = list(histories)
for name, marker in [("Point Jacobi", "o"), ("Point SGS", "s"), ("Patch Jacobi", "^"),
                     ("Patch SGS", "D"), ("V-cycle", "v")]:
    label = "CG / point Jacobi" if name == "Point Jacobi" else name
    residuals = histories[sizes[-1]][name]
    line, = ax_res.semilogy(residuals, label=label)
    steps = [len(histories[N][name]) - 1 for N in sizes]
    ax_it.semilogx(sizes, steps, marker=marker, color=line.get_color(), label=label)

ax_res.axhline(tolerance, color="0.4", linestyle=":", linewidth=1)
ax_res.set(xlabel="Iteration", ylabel=r"$\|b-AU^{(k)}\|_2/\|b\|_2$",
           title=f"Finest mesh ({sizes[-1]} free dofs)")
ax_it.set(xlabel="Free degrees of freedom", ylabel="PCG iterations",
          title="Uniform mesh refinement", xticks=sizes, xticklabels=sizes)
ax_it.legend(fontsize=9)
for ax in (ax_res, ax_it):
    ax.grid(alpha=0.25)
plt.show()

Further reading: [NGSolve, Building blocks for programming preconditioners](https://docu.ngsolve.org/latest/i-tutorials/unit-2.1.2-blockjacobi/blockjacobi.html)